This notebook contains the code used to compile the results from the raw data.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

from utils import validate_data_split_params
from sklearn.linear_model import RidgeCV
from tqdm import tqdm

In [ ]:
def get_splits(mg, states, data_split_params):
    N_train = data_split_params['train_phase']['number_train_timesteps']
    gap = data_split_params['train_phase']['gap']

    h_max = data_split_params['test_phase']['prediction_horizon']
    N_test_warmup = data_split_params['test_phase']['number_warmup_timesteps']
    N_o = data_split_params['test_phase']['number_forecast_origins']
    s = data_split_params['test_phase']['forecast_origin_spacing']
    rolling_window = N_o*(1+s)-s

    # Train split
    X_train = mg[:, :N_train+h_max]
    train_states = states[:N_train+h_max, :]
    y_train = np.zeros((N_train, h_max)) # Each column corresponds to h = 1, 2, ... pred_horizon

    for i in range(N_train):
        y_train[i, :] = np.roll(X_train[1, :].flatten(), shift=-1-i)[:h_max].flatten()

    # Test split
    X_test_warmup = mg[:, N_train+h_max+gap:N_train+h_max+gap+N_test_warmup]
    X_test = mg[:, N_train+h_max+gap+N_test_warmup:N_train+h_max+gap+N_test_warmup+rolling_window+h_max]
    test_states = states[N_train+h_max+gap+N_test_warmup:N_train+h_max+gap+N_test_warmup+rolling_window, :]

    y_test = np.zeros((rolling_window, h_max))
    for i in range(rolling_window):
        y_test[i, :] = np.roll(X_test[1, :].flatten(), shift=-1-i)[:h_max].flatten()

    X_test = X_test[:, :rolling_window]

    return X_train, y_train, X_test_warmup, X_test, y_test, train_states, test_states

In [ ]:
root_paths = [
    r'data\raw\part1',
    r'data\raw\part2',
]

results_path = [os.path.join(rpath, i) 
                for rpath in root_paths 
                for i in os.listdir(rpath)]

results = []

for path in results_path:
    if not path.endswith('.npz'):
        continue
    r = np.load(path, allow_pickle=True)['results'].tolist()
    r[0]['run_folder'] = path.split("\\")[-2]
    r[0]['file_name'] = path.split("\\")[-1]
    results.extend(r)

df = pd.DataFrame(results)

In [ ]:
seed2runID_map = {seed: idx for idx, seed in enumerate(df['train_phase_seed'].unique())}

df['runID'] = df['train_phase_seed'].apply(lambda x: seed2runID_map[x])

len(seed2runID_map)

In [ ]:
df[['optical_features', 'runID', 'run_folder', 'file_name']].sort_values(by=['runID', 'optical_features'])

In [ ]:
mg = []

states_lin = []
states_nonlin = []

X_train = []
y_true = []
y_lin = []
y_nonlin = []

corr_lin = []
corr_nonlin = []

corr_ts_min = np.inf

results = {}

for seed, runID in seed2runID_map.items():

    try:
        df_lin = df[(df['runID'] == runID) & (df['optical_features'] == 'linear')]
        df_nonlin = df[(df['runID'] == runID) & (df['optical_features'] == 'nonlinear')]

        assert len(df_lin) == 1, f'RunID: {runID} - Missing linear experiment'
        assert len(df_nonlin) == 1, f'RunID: {runID} - Missing nonlinear experiment'
        assert (df_lin['X_train'].iloc[0] == df_nonlin['X_train'].iloc[0]).all(), 'X_train_lin != X_train_nonlin'
        assert (df_lin['y_train'].iloc[0] == df_nonlin['y_train'].iloc[0]).all(), 'y_train_lin != y_train_nonlin'
        assert (df_lin['X_test'].iloc[0] == df_nonlin['X_test'].iloc[0]).all(), 'X_test_lin != X_test_nonlin'
        assert (df_lin['y_test'].iloc[0] == df_nonlin['y_test'].iloc[0]).all(), 'y_test_lin != y_test_nonlin'


        corr_lin.append(df_lin['corr_lin'].iloc[0])
        corr_nonlin.append(df_nonlin['corr_nonlin'].iloc[0])

        dt = df_lin['X_test'].iloc[0][0, 1] - df_lin['X_test'].iloc[0][0, 0]
        X_test = np.hstack((df_lin['X_test'].iloc[0][1, :], df_lin['y_test'].iloc[0][-1, :]))
        X_test = np.vstack((np.arange(df_lin['X_test'].iloc[0][0, 0], df_lin['X_test'].iloc[0][0, 0]+len(X_test))*dt, X_test))
        
        mg.append(np.concat((df_lin['X_train'].iloc[0], X_test), axis=1)) # Same for both lin and nonlin
        X_train.append(df_lin['X_train'].iloc[0]) # Same for both lin and nonlin
        y_true.append(df_lin['y_test'].iloc[0]) # Same for both lin and nonlin
        y_lin.append(df_lin['reservoir_predictions_multi'].iloc[0])
        y_nonlin.append(df_nonlin['reservoir_predictions_multi'].iloc[0])

        states_lin.append(np.concat((df_lin['train_states'].iloc[0], df_lin['test_states'].iloc[0]), axis=0))

        states_nonlin.append(np.concat((df_nonlin['train_states'].iloc[0], df_nonlin['test_states'].iloc[0]), axis=0))

        corr_ts_min = int(np.min([corr_ts_min, len(df_lin['corr_lin'].iloc[0]), len(df_nonlin['corr_nonlin'].iloc[0])]))

        # if runID == 1:
        #     break
    
    except Exception as e:
        print(e)

results[30000] = {
    'y_true': np.array(y_true),
    'y_lin': np.array(y_lin),
    'y_nonlin': np.array(y_nonlin),
    'X_train': np.array(X_train),
    'corr_lin': np.array([c[:corr_ts_min] for c in corr_lin]),
    'corr_nonlin': np.array([c[:corr_ts_min] for c in corr_nonlin])
}

mg = np.array(mg)
states_lin = np.array(states_lin)
states_nonlin = np.array(states_nonlin)

results[30000]['y_true'].shape # Dimension: (trial number, samples, prediction horizon)

In [ ]:
# Because we are using the multi-step ahead forecasting strategy, from a long run of 30k points
# we can use only a few of them and check the reservoir performance with less points.
# Indeed, we can show that the reservoir performance after 10k plateaus (see the scaling law plot in figures.ipynb)

for N_train in [10000]: # [1000, 1500, 2000, 3000, 4000, 5000, 7500, 10000, 12500, 15000, 20000, 25000]:
    data_split_params = {
            'train_phase': {
                'x0': None,
                'seed': None,
                'forget': 100,
                'number_train_timesteps': N_train,
                'gap': 0
            },
            'test_phase': {
                'prediction_horizon': 500,
                'number_forecast_origins': 5000,
                'forecast_origin_spacing': 0,
                'number_warmup_timesteps': 0
            }
        }
    validate_data_split_params(data_split_params)

    y_true = []
    y_lin = []
    y_nonlin = []
    X_train = []

    for expID in tqdm(range(14)): # N_exps
        
        for opt_feature in ['linear', 'nonlinear']:

            states = states_lin if opt_feature == 'linear' else states_nonlin
            
            forget = data_split_params['train_phase']['forget']
            h_max = data_split_params['test_phase']['prediction_horizon']

            X_train_temp, y_train, X_test_warmup, X_test, y_test, train_states, test_states = get_splits(mg=mg[expID, :, :], states=states[expID, :, :], data_split_params=data_split_params)
            reg_model = RidgeCV(**{'fit_intercept': True, 'alphas': np.logspace(-8, 0), 'alpha_per_target': True, 'store_cv_results': False})
            reg_model.fit(train_states[forget:-h_max], y_train[forget:, :])
            W_out = reg_model.coef_.T
            b_out = reg_model.intercept_

            if opt_feature == 'linear':
                y_lin.append(test_states @ W_out + b_out)
            else:
                y_nonlin.append(test_states @ W_out + b_out)

        y_true.append(y_test)
        X_train.append(X_train_temp)

    results[N_train] = {
            'y_true': np.array(y_true),
            'y_lin': np.array(y_lin),
            'y_nonlin': np.array(y_nonlin),
            'X_train': np.array(X_train)
        }

In [ ]:
joblib.dump(results, r'data\results.joblib')